In [1]:
#!pip install pymongo

In [1]:
from flask import Flask, request, jsonify  
import pymongo
from pymongo import MongoClient
from bson.objectid import ObjectId
import threading  
import time  

In [2]:
app = Flask(__name__)

In [3]:
client = MongoClient("mongodb://localhost:27017/")
db = client["CustomersDB"]
customers_col = db["Customers"]

In [4]:
# === Required Libraries ===
from flask import Flask, request, jsonify  # Flask for API, request for client data, jsonify to return JSON
from pymongo import MongoClient  # MongoClient to connect to MongoDB
from bson.objectid import ObjectId  # ObjectId for working with MongoDB IDs
import threading  # For background server thread
import time  # Sleep to allow Flask to start

# === Initialize Flask App ===
app = Flask(__name__)  # Create Flask app instance

# === MongoDB Connection Setup ===
client = MongoClient("mongodb://localhost:27017/")  # Connect to MongoDB running locally
db = client["CustomersDB"]  # Use or create "CustomersDB" database
customers_col = db["Customers"]  # Use or create "Customers" collection

# === GET /customers: Return all customers ===
@app.route('/customers', methods=['GET'])
def get_customers():
    # Retrieve all customers from MongoDB and convert ObjectId to string
    customers = []
    for doc in customers_col.find():
        doc['_id'] = str(doc['_id'])  # Convert ObjectId to string for JSON
        customers.append(doc)
    return jsonify(customers)

# === POST /customers: Add a new customer ===
@app.route('/customers', methods=['POST'])
def add_customer():
    data = request.json  # Parse input JSON
    try:
        # Insert document into MongoDB collection
        result = customers_col.insert_one({
            "Name": data['Name'],
            "Age": data['Age'],
            "Gender": data['Gender'],
            "Region": data['Region']
        })
        return jsonify({'message': 'Customer added', 'id': str(result.inserted_id)}), 201
    except Exception as e:
        return jsonify({'error': str(e)}), 400

# === PUT /customers/<id>: Update customer by MongoDB _id ===
@app.route('/customers/<string:id>', methods=['PUT'])
def update_customer(id):
    data = request.json
    try:
        # Update the document in MongoDB
        result = customers_col.update_one(
            {"_id": ObjectId(id)},
            {"$set": {
                "Name": data['Name'],
                "Age": data['Age'],
                "Gender": data['Gender'],
                "Region": data['Region']
            }}
        )
        if result.matched_count == 0:
            return jsonify({'error': 'Customer not found'}), 404
        return jsonify({'message': 'Customer updated'})
    except Exception as e:
        return jsonify({'error': str(e)}), 400

# === DELETE /customers/<id>: Delete customer by MongoDB _id ===
@app.route('/customers/<string:id>', methods=['DELETE'])
def delete_customer(id):
    try:
        result = customers_col.delete_one({"_id": ObjectId(id)})
        if result.deleted_count == 0:
            return jsonify({'error': 'Customer not found'}), 404
        return jsonify({'message': 'Customer deleted'})
    except Exception as e:
        return jsonify({'error': str(e)}), 400

# === Run Flask App in a Background Thread (Jupyter-friendly) ===
def run_flask():
    app.run(port=5000, debug=False, use_reloader=False)  # Don't reload so thread doesn't restart

threading.Thread(target=run_flask).start()  # Start Flask in background
time.sleep(1)  # Wait for server to boot up

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit


In [5]:
import requests
res = requests.get("http://127.0.0.1:5000/customers")
print("GET:", res.json())

GET: [{'Age': 20, 'Gender': 'Male', 'Name': 'Ahmed', 'Region': 'Cairo', '_id': '68974ff636e21f42f20203f9'}, {'Age': 30, 'Gender': 'Male', 'Name': 'Sara', 'Region': 'Cairo', '_id': '689756b10d3d9e2ad3db8c42'}]


In [6]:
response = requests.post("http://127.0.0.1:5000/customers", json={
    "Name": "Alice Smith",
    "Age": 30,
    "Gender": "Female",
    "Region": "North"
})
print("POST:", response.json())
#customer_id = response.json().get("id")  # Get the actual ObjectId from the response for later use
#print("New customer ID:", customer_id)

POST: {'id': '689768140d3d9e2ad3db8c43', 'message': 'Customer added'}


In [8]:
cust_id = '689768140d3d9e2ad3db8c43'
response = requests.put(f"http://127.0.0.1:5000/customers/{cust_id}", json={
    "Name": "Alice Jones",
    "Age": 31,
    "Gender": "Female",
    "Region": "East"
})
print("PUT:", response.json())

JSONDecodeError: Expecting value: line 1 column 1 (char 0)